# Technical report: Distilling Context into Parameters for Time-Series Foundation Models for Transportation Forecasting

This report serves as the main technical report for the project *Distilling Context into Parameters for Time-Series Foundation Models for Transportation Forecasting* authored by Felix Thomsen (s221710), Christian Rand (s224930), Alexander Schiøtz (s221221), Bertram Hage (s224918) as part of the DTU course 42578 Advanced Business Analytics 2026.

The following report outlines the complete methodology and framwork design and reports the full results. For code implementation we refer to the seperate notebook [code_implementation.ipynb](code_implementation.ipynb). For post-hoc analysis code we refer to the files under `post_hoc/`.

**Table of contents:**
- Background and previous work <!-- Done -->
- Preliminaries <!-- Done -->
- Methology and data <!-- Done, missing data -->
- Experimental setup <!-- TODO -->
- Results and discussion <!-- TODO -->
- References <!-- Done -->

## Background and previous work
The ability to accurately predict traffic flow is essential for building resilient urban mobility systems capable of adapting to sudden disruptions and maintaining efficiency under stress. In the paper "Time series foundation models as strong baselines in transportation forecasting: A large-scale benchmark analysis" J. Pulido and F. Rodrigues shows that the foundational transformer-based timeseries model Chronos-2 by Amazon achieves state-of-the-art forecasting accuracy in a zero-shot capacity accross various datasets in the traffic domain [[1]](https://arxiv.org/pdf/2602.24238). While this is promising for the adaptation of foundational models application in traffic speed predictions, a significant limitation of these models is the need to pass long context windows, which forces the architecture to redundantly re-learn the inherent traffic patterns at each inference step. Since latency and memory for transformer models scales with a time complexity of $O(n^2)$ with context length, this increases the hardware requirements for deployments of such models in a real-world setting limiting their usability.

In "Doc-to-lora: Learning to instantly internalize contexts" R. Charakorn, E. Cetin, S. Uesaka, and R. T. Lange propose an approach for distilling long textual contexts to LoRA adapters for a foundational large language model (LLM) using a learned hypernetwork, in a single forward pass [[2]](https://arxiv.org/pdf/2602.15902). The hypernetwork is trained to minimize the KL-divergence between the teacher LLMs response with the full context and LoRA-adapted student LLMs response, recieving only a short query. On a needle-in-a-haystack task, the approach successfully maps contexts to adapters, achieving near-perfect zero-shot accuracy on sequences over 4 times the LLM's native window.


## Preliminaries
This section briefly outlines preliminary theory used in this project.

### LoRA adapters
LoRA (Low-Rank Adaptation) adapters are small, trainable modules added to a pre-trained neural network to adapt it to a new task without updating most of the original weights [[3]](https://arxiv.org/pdf/2106.09685). Instead of fine-tuning full weight matrices, LoRA learns a low-rank update that approximates changes in weights. 
$$
\Delta W = B A,
$$
with $W \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$ and $A \in \mathbb{R}^{r \times d_{\text{in}}}, \qquad B \in \mathbb{R}^{d_{\text{out}} \times r}, \qquad r \ll \min(d_{\text{in}}, d_{\text{out}})$.

This makes training much cheaper and lets you swap or combine task-specific adapters while keeping the same frozen base model.

### Chronos-2 foundational model
Chronos-2 is a pre-trained encoder-only Transformer model for time-series forecasting developed by Amazon [[4]](https://arxiv.org/pdf/2510.15821). It first splits the input time series into fixed-length patches (segments), embeds them, and processes the resulting patch sequence with a stack of attention blocks.

Internally, the architecture alternates between (i) time attention, which applies self-attention along the temporal axis (using rotary position embeddings, RoPE), and (ii) group attention, which aggregates information across related time series within the same group (defined by group IDs) at a given patch index. In our project we do not utilize group attention, and it is a topic for future research how to integrate group attention with our LoRA-based context destillation.

Finally, Chronos-2 produces probabilistic multi-step forecasts with a quantile head that predicts a grid of quantiles for each forecast horizon step.

## Methology and data
Inspired by the approach by R. Charakorn and collegues we in this paper transfer the idea to a timeseries prediction task. Specifically, we investigate the application of the approach on the PEMS-BAY dataset, with data from San Francisco’s highway system, consisting of cleaned data from 325 under-road sensors used to train AI models to anticipate real-world traffic jams. A more thorough description of the data is presented in the following section.

### Data
The dataset utilized for training the hypernetwork and evaluating the distilled adapters is the PEMS-BAY traffic speed dataset [[5]](https://www.kaggle.com/datasets/scchuy/pemsbay). Provided by the California Department of Transportation, this dataset features traffic speed readings collected from 325 distinct sensors situated throughout the San Francisco Bay Area. The data spans a timeframe from January 1st 2017 to June 30th 2017, and is recorded at a 5-minute sampling granularity. This high-frequency structure provides a rigorous testbed for time-series foundation models, as it requires the model to internalize both repetitive weekly patterns and hard to predict localized disruptions.



### High level design
The goal is to have a hypernetwork that, once trained, is able to accept varied length context of a single station and output a useful LoRA adapter. This LoRA adapter can then be applied to the foundational Chronos-2 model and together with a short context window accuratly predict traffic speeds at a horizon of 60 minutes. Once the LoRA adapter has been outputtet by the hypernetwork for a station it can be used for various short context time windows where the time window appears after the long context.


![High-level](/Users/bertramhage/DTU/advanced-ba/project/assets/img/high_level_mermaid.png)

Formally, let $\mathcal{D}$ be a multivariate time-series dataset comprising $N$ stations observed over $T$ timesteps. For a given station $i \in \{1, \ldots, N\}$ and a forecast origin $t$, the task is to predict the sequence of future values over a forecast horizon $H$, corresponding to the time indices $\{t+1, t+2, \ldots, t+H\}$.

To generate this prediction, the proposed architecture relies on two distinct observation windows:

- Short Context ($C_{short}$): A recent historical window of length $W_{short}$ that immediately precedes the forecast origin. It is defined as the set of indices $\{t - W_{short} + 1, \ldots, t\}$. This serves as the direct input to the LoRA-adapted foundation model during online inference.
- Long Context ($C_{long}$): An extended historical window of length $W_{long}$ utilized offline by the hypernetwork to distill the station's temporal dynamics into adapter parameters. This window spans a set of indices $\{t_{start}, \ldots, t_{end}\}$ such that $t_{end} < t - W_{short} + 1$. Thus, the short context may immediately preceed the long context or there may be a time gap in between. In our implementation $C_{long}$ is fixed for all forecast horizons $t$.

![Time scale](/Users/bertramhage/DTU/advanced-ba/project/assets/img/time_scale.png)

### Hypernetwork design

The goal of the hypernetwork is to compile a long historical window for a single station into a small set of LoRA parameters that can later be reused for many short-context forecasts. Concretely, we learn a mapping

$$
 h_\phi: C_{\text{long}} \;\mapsto\; \{\Delta W^{(\ell,m)}\}_{\ell=1..12,\;m\in\{q,k,v,o\}},
$$

where each $\Delta W^{(\ell,m)}$ is represented by a low-rank factorization (LoRA) for Chronos-2’s time self-attention projections: query ($q$), key ($k$), value ($v$), and output ($o$) in each of the 12 encoder blocks.

#### Context encoder
We first embed the long context with a frozen encoder. For this we use the first 8 layers of the Chronos-2 model. Given the raw time-series values of shape $[1, W_{\text{long}}]$, the context encoder returns the last hidden states

$$Z \in \mathbb{R}^{1 \times S \times 768},$$

where $S$ is the number of context patches produced by Chronos-2. Keeping this encoder reduces the number of parameters needed to be learned by the hypernetwork, and we assume the initial layers of the Chronos-2 model already has learned a valuable representation of time-series data.

#### Perceiver aggregator
The number of context patches $S$ can vary, so we need a module that turns a variable-length sequence $Z$ into a fixed-size representation. Inspired by Doc-to-LoRA, we use a Perceiver-style aggregator: a small set of learned latent query-vectors cross-attend to $Z$ and collect the relevant information. We use 32 latent queries in our implementation.

The Perceiver outputs a fixed set of $N_{\text{out}}=384$ vectors, computed as $12\ \text{layers} \times 4\ \text{modules} \times \text{LoRA rank}=8$, each of size 128. This can be thought of as one vector per layer, per module, per rank-slot.

#### Projection to LoRA weights
Finally, each 128-dimensional vector is passed through a 1-layer residual MLP and then through a layer/module-specific linear head that outputs the two LoRA matrices for each layer $\ell$ and module $m$

$$A^{(\ell,m)} \in \mathbb{R}^{r \times d_{\text{model}}}, \qquad B^{(\ell,m)} \in \mathbb{R}^{d_{\text{model}} \times r},$$

with $r=8$ and $d_{\text{model}}=768$.

### Training setup

The training objective is to learn a hypernetwork $h_\phi$ that maps a long context window $C_{\text{long}}$ for a single station to LoRA weights for Chronos-2. During training, the frozen Chronos-2 backbone is never updated, only the hypernetwork parameters are optimized.

The training setup follows the high-level structure:

1. Sample a batch of long-context windows. A sample corresponds to a specific station and a start index of a rolling window in time.
2. For each long context, we select $Q$ forecast origins spaced by a fixed stride within the admissible range. This yields $Q$ short-context windows $\{C_{\text{short}}^{(q)}\}_{q=1..Q}$ and associated targets, all sharing the same long context. The hypernetwork must therefore produce an adapter that performs well across several forecast origins, rather than overfitting to a single moment. This is analogous to the approach in Doc-to-LoRA where multiple text-queries where generated for a single piece of long context.
3. For each sample, encode long context and generate LoRA weights according to the hypernetwork design described above.
4. The generated LoRA weights are injected into Chronos-2, and the LoRA-adapted model is run on each short-context query to produce probabilistic forecasts (quantile predictions) for the horizon $H$.
5. The loss is averaged over the batch, the $Q$ queries per long context, and the $H$ forecast steps, and gradients flow only back into the hypernetwork.

For the training of the hypernetwork we experiment with two training objectives.

#### Training with distillation targets

Here we define:

- **Teacher:** frozen Chronos-2 with the full available $C_{\text{long}}$ followed by $C_{\text{short}}$ passed.
- **Student:** frozen Chronos-2 augmented with the LoRA adapter produced from the hypernetwork and $C_{\text{long}}$, run only on the short-context $C_{\text{short}}$.

Let the teacher and student quantile outputs
$$
\hat{Q}^{\text{(T)}} \in \mathbb{R}^{Q \times K \times H}, \qquad \hat{Q}^{\text{(S)}} \in \mathbb{R}^{Q \times K \times H},
$$
where $K$ is the number of quantile levels and $H$ is the prediction length, we minimize:
$$
\mathcal{L}_{\text{teacher}} = \frac{1}{QKH}\sum_{q=1}^{Q}\sum_{k=1}^{K}\sum_{h=1}^{H} \operatorname{SmoothL1}\big(\hat{Q}^{\text{(S)}}_{q,k,h} - \hat{Q}^{\text{(T)}}_{q,k,h}\big),
$$
where
$$
\operatorname{SmoothL1}(x) = 
\begin{cases} 
0.5 x^2 & \text{if } |x| < 1 \\
|x| - 0.5 & \text{otherwise}
\end{cases}.
$$

This objective encourages the LoRA-adapted student to match the full-context probabilistic forecast of the teacher. 
#### Training with ground truth targets

Here we remove the teacher entirely and supervise the student directly on the ground truth future values $y \in \mathbb{R}^{Q \times H}$. Since Chronos-2 predicts quantiles, we use a loss computed from quantile losses.

For a quantile level $\tau \in (0,1)$ and error $u = y - \hat{q}_{\tau}$, the quantile loss is
$$
\rho_{\tau}(u) = \max\big(\tau u, (\tau - 1)u\big).
$$
With quantile levels $\{\tau_k\}_{k=1..K}$, we compute the combined loss as the average quantile loss across quantiles:
$$
\mathcal{L}_{\text{gt}} = \frac{2}{QKH}\sum_{q=1}^{Q}\sum_{k=1}^{K}\sum_{h=1}^{H} \rho_{\tau_k}\big(y_{q,h} - \hat{Q}^{\text{(S)}}_{q,k,h}\big).
$$

By scaling the loss with 2 the interpretation becomes close to that of the mean-absolute-error.

This objective directly optimizes forecasting quality against the data, while still training only the hypernetwork parameters. 

#### Multi-query sampling and hierarchical length jitter

To increase the generalization of our hypernetwork we add Gaussian noise to the length of the long and short context windows during training. This is done in a hierarchical manner to keep batches memory efficient. 

For each batch we first sample a batch-level mean length from a Gaussian distribution, then sample per-sample lengths around that mean, again from a Gaussian distribution. 

During initial validation runs this greatly increased the generalizability of the hypernetwork and the downstream evaluation performance.

## References
[1] J. Pulido and F. Rodrigues, “Time series foundation models as strong baselines in transportation forecasting: A large-scale benchmark analysis,” *arXiv preprint arXiv:2602.24238*, 2026.\
[2] R. Charakorn, E. Cetin, S. Uesaka, and R. T. Lange, “Doc-to-lora: Learning to instantly internalize contexts,” *arXiv preprint arXiv:2602.15902*, 2026\
[3] E. J. Hu, Y. Shen, P. Wallis, Z. Allen-Zhu, Y. Li, S. Wang, and W. Chen, “Lora: Low-rank adaptation of large language models,” *CoRR*, vol.abs/2106.09685, 2021\
[4] A. F. Ansari, O. Shchur, J. K¨uken, A. Auer, B. Han, P. Mercado, S. S.
Rangapuram, H. Shen, L. Stella, X. Zhang, M. Goswami, S. Kapoor,
D. C. Maddix, P. Guerron, T. Hu, J. Yin, N. Erickson, P. M. Desai,
H. Wang, H. Rangwala, G. Karypis, Y. Wang, and M. Bohlke-Schneider,
“Chronos-2: From univariate to universal forecasting,” 2025
[5] S. Sun, "PEMS-BAY," Kaggle, 2023. [Online]. Available: https://www.kaggle.com/datasets/scchuy/pemsbay